# Ensemble weights: size-aware, RT-DETR up-weighted, new 1280 members, and Faster R-CNN as a score-only vote

`ensemble_temporal.ipynb` fused all members with **equal weights**. Its best was Person B's 7 models + RetinaNet (WBF IoU 0.65, val mAP 0.2998). Adding Faster R-CNN raised AP_small but lowered mAP and AP75, which suggests its boxes are less precise and pull the fused boxes off target.

**What the public leaderboard showed (2026-11-08):** size-aware weights (p = 2) scored 0.3268 against 0.3145 for equal weights (val: +0.003, LB: +0.012), and Person B's 7 models with RT-DETR weighted ×2 scored 0.3278, the best so far, although its val mAP was lower (0.2967). RT-DETR alone scores 0.301 on the leaderboard, so on test it carries most of the ensemble. This version therefore adds an **RT-DETR boost** (every RT-DETR member's weight × 1.5 / 2 / 3, on top of the size-aware weights) and the two models trained overnight (`train_overnight.ipynb`): **RT-DETR-L 1280** (val 0.2697, the best single model) and **YOLO26m 1280**, both trained on original + test-like frames.

Here, on the cached member predictions (CPU only, no model is re-run):

| Scheme | What changes |
|---|---|
| plain | equal weights (reference) |
| RT-DETR boost b | every RT-DETR member's weight × b (alone, or on top of size-aware) |
| size-aware, p | weight per model **and box size** (small < 32², medium < 96², large, in original pixels): `w = (AP_size,model / mean AP_size) ^ p`. YOLO at 1280 is best on small objects, RT-DETR on large ones |
| Faster R-CNN vote | fuse the other models, then Faster R-CNN only changes the **scores** of fused boxes it agrees with (IoU ≥ 0.55), never their coordinates; optionally its unmatched boxes are added with a low score |

WBF multiplies every box's score by its weight, and that product sets both the fused score and how much the box pulls the fused position, so a per-box weight is applied by scaling scores before `fuse`.

**Overfitting check:** each scheme has a single tuned number (p or the vote weight), not one weight per model; the best scheme is also compared clip by clip against plain.

**Run after** `ensemble_temporal.ipynb` (all members' val and test predictions cached). About 45–60 min of CPU. Outputs: `runs/ensemble/weights_results_final.csv` (yesterday's grid stays in `weights_results.csv`), `weights_best.json`, the best fusion on every val and test frame (`fused_best_valall.pkl`, `fused_best_test.pkl`, used by `temporal_v2.ipynb`), and `submissions/W_final.csv`.

## Setup

In [1]:
import json
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path.cwd().parent            # this notebook lives in src/experiments/
ROOT = SRC.parent
sys.path.insert(0, str(SRC))
from common.evaluate import evaluate_detections
from common.pipeline import fuse, to_submission_rows
from common.submission import write_submission

DATA_ROOT = ROOT / "multiple-object-video-detection-hw-2-ucu-ai-2026"
GT_FILE = ROOT / "data_processed" / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
RUNS = ROOT / "runs"
ENS = RUNS / "ensemble"

val_meta = json.loads((DATA_ROOT / "val" / "annotations_coco.json").read_text())["images"]
VAL = [{"image_id": im["id"], "video": im["video"], "frame": im["frame_id"], "shape": (im["height"], im["width"])}
       for im in val_meta]
test = pd.read_csv(TEST_CSV)
TEST = [{"image_id": k, "video": c, "frame": int(f), "shape": (h, w)}
        for k, c, f, w, h in zip(test["image_id"], test["clip"], test["frame"], test["width"], test["height"])]
LABELLED = sorted(im["id"] for im in json.loads(GT_FILE.read_text())["images"])
VAL_BY_ID = {f["image_id"]: f for f in VAL}
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]

final_b = json.loads((RUNS / "pipeline" / "final_config.json").read_text())
B_MEMBERS = list(final_b["members"])
FRCNN, RETINA = "frcnn_baseline_ms", "retinanet_baseline_ms"
RT1280, Y26M = "rtdetr_l_1280_testres", "yolo26m_1280_testres"   # trained overnight
WEAK = ["rtdetr_960", "yolo_final_960"]          # the two weakest members on val (mAP ~0.19-0.23 with flip)
ALL = B_MEMBERS + [FRCNN, RETINA, RT1280, Y26M]
preds = {n: pickle.loads((ENS / f"valall_{n}.pkl").read_bytes()) for n in ALL}
print(f"{len(ALL)} members loaded | {len(LABELLED)} labelled val frames")

11 members loaded | 359 labelled val frames


## 1. Each member's AP by object size

The size-aware weights come from this table (labelled val frames).

In [2]:
def to_dets(boxes_by_id, ids):
    dets = []
    for i in ids:
        xyxy, s, c = boxes_by_id[i]
        dets += [{"image_id": i, "category_id": int(k) + 1, "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                  "score": float(v)} for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
    return dets


member_ap = pd.DataFrame({n: evaluate_detections(GT_FILE, to_dets(preds[n], LABELLED), image_ids=LABELLED) for n in ALL}).T
member_ap[COLS].round(4)

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR
yolo_low_scale_960,0.2206,0.4145,0.2067,0.1077,0.1772,0.4036,0.3562
yolo_baseline_1280,0.2204,0.4343,0.1994,0.1527,0.2145,0.3479,0.3683
yolo_final_960,0.1951,0.3778,0.1814,0.1056,0.1748,0.3305,0.3236
yolo_final_1280,0.1984,0.3988,0.1747,0.1097,0.1961,0.3274,0.3271
rtdetr_960,0.2341,0.4480,0.2173,0.0912,0.1975,0.4114,0.3899
rtdetr_final_960,0.2486,0.4774,0.2260,0.1103,0.2182,0.4243,0.4050
rtdetr_final_960_s025,0.2534,0.4891,0.2310,0.1089,0.2162,0.4150,0.4368
frcnn_baseline_ms,0.2192,0.4622,0.1809,0.1101,0.2017,0.4057,0.4586
retinanet_baseline_ms,0.1913,0.3753,0.1724,0.0921,0.1793,0.3535,0.3983
rtdetr_l_1280_testres,0.2798,0.5304,0.2583,0.1248,0.2417,0.4720,0.4264


## 2. Weighting schemes

In [3]:
SIZES = ["AP_small", "AP_medium", "AP_large"]


def size_bin(xyxy):
    area = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    return np.digitize(area, [32 ** 2, 96 ** 2])   # 0 small, 1 medium, 2 large (COCO thresholds)


def weights_for(members, scheme, p, boost=1.0):
    # {member: array of 3 weights (small, medium, large)}; boost multiplies every RT-DETR member
    ap = member_ap.loc[members]
    if scheme == "plain" or p == 0:
        w = {n: np.ones(3) for n in members}
    elif scheme == "global":
        g = (ap["mAP"] / ap["mAP"].mean()) ** p
        w = {n: np.full(3, g[n]) for n in members}
    else:
        rel = (ap[SIZES] / ap[SIZES].mean()).fillna(1.0)          # a size with no ground truth (AP undefined): weight 1
        w = {n: np.maximum(rel.loc[n].to_numpy(), 0.05) ** p for n in members}   # no member is switched off entirely
    return {n: v * (boost if "rtdetr" in n else 1.0) for n, v in w.items()}


def box_iou(a, b):
    # IoU between xyxy boxes a (N, 4) and b (M, 4)
    iw = (np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0])).clip(0)
    ih = (np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1])).clip(0)
    inter = iw * ih
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)


def frcnn_vote(fused, fr, n, beta, add):
    # Faster R-CNN as one extra voter of weight beta that changes scores only (WBF 'avg': mean over n + beta voters)
    xyxy, s, c = fused
    fx, fs, fc = fr
    new = s * n / (n + beta)
    used = np.zeros(len(fx), bool)
    for k in np.unique(c):
        a, b = np.where(c == k)[0], np.where(fc == k)[0]
        if not len(b):
            continue
        iou = box_iou(xyxy[a], fx[b])
        j = iou.argmax(1)
        hit = iou[np.arange(len(a)), j] >= 0.55
        new[a[hit]] += beta * fs[b[j[hit]]] / (n + beta)
        used[b[(iou >= 0.55).any(0)]] = True
    if add and (~used).any():
        xyxy = np.concatenate([xyxy, fx[~used]])
        new = np.concatenate([new, beta * fs[~used] / (n + beta)])
        c = np.concatenate([c, fc[~used]])
    order = np.argsort(-new)[:500]
    return xyxy[order], new[order].astype(np.float32), c[order]


def fuse_frame(cfg, preds, image_id, shape):
    w = weights_for(cfg["members"], cfg["scheme"], cfg["p"], cfg.get("boost", 1.0))
    per = []
    for n in cfg["members"]:
        xyxy, s, c = preds[n][image_id]
        per.append((xyxy, s * w[n][size_bin(xyxy)], c))
    fused = fuse(per, shape, iou=cfg["iou"])
    if cfg.get("vote"):
        fused = frcnn_vote(fused, preds[FRCNN][image_id], len(cfg["members"]), **cfg["vote"])
    return fused

## 3. Grid on the labelled val frames

Member sets: Person B's 7 (to reproduce the leaderboard-best idea, RT-DETR ×2), B's 7 + RetinaNet (yesterday's best), the same + RT-DETR 1280, + YOLO26m too, and without the two weakest members. Schemes: equal weights, size-aware p = 2 (yesterday's 0.3268 on the leaderboard), and size-aware with the RT-DETR boost. WBF IoU 0.65 throughout. Then the Faster R-CNN score-only vote is tried on the best configuration.

In [4]:
SETS = {
    "B7": B_MEMBERS,
    "B7+Ret": B_MEMBERS + [RETINA],
    "B7+Ret+RT1280": B_MEMBERS + [RETINA, RT1280],
    "B7+Ret+RT1280+Y26m": B_MEMBERS + [RETINA, RT1280, Y26M],
    "B7+Ret+RT1280 -weak": [n for n in B_MEMBERS if n not in WEAK] + [RETINA, RT1280],
}
SCHEMES = {"plain": dict(scheme="plain", p=0), "plain, RT-DETR x2": dict(scheme="plain", p=0, boost=2.0),
           "size p=2": dict(scheme="size", p=2), "size p=2, RT-DETR x1.5": dict(scheme="size", p=2, boost=1.5),
           "size p=2, RT-DETR x2": dict(scheme="size", p=2, boost=2.0), "size p=2, RT-DETR x3": dict(scheme="size", p=2, boost=3.0)}
CONFIGS = {}
for set_name, members in SETS.items():
    for scheme_name, sch in SCHEMES.items():
        if set_name == "B7" and scheme_name not in ("plain", "plain, RT-DETR x2", "size p=2, RT-DETR x2"):
            continue
        CONFIGS[f"{set_name} | {scheme_name}"] = dict(members=members, iou=0.65, **sch)


def run_config(label, cfg, rows):
    t0 = time.time()
    fused = {i: fuse_frame(cfg, preds, i, VAL_BY_ID[i]["shape"]) for i in LABELLED}
    rows[label] = evaluate_detections(GT_FILE, to_dets(fused, LABELLED), image_ids=LABELLED)
    print(f"{label:50s} mAP {rows[label]['mAP']:.4f}  AP_small {rows[label]['AP_small']:.4f}  ({time.time() - t0:.0f} s)", flush=True)


rows = {}
for label, cfg in CONFIGS.items():
    run_config(label, cfg, rows)
# Faster R-CNN as a score-only vote, on top of the best configuration so far
best_label = max(rows, key=lambda k: rows[k]["mAP"])
for beta in (0.5, 1.0):
    for add in (False, True):
        label = f"{best_label} + FRCNN vote {beta}{' +add' if add else ''}"
        CONFIGS[label] = dict(CONFIGS[best_label], vote=dict(beta=beta, add=add))
        run_config(label, CONFIGS[label], rows)

grid = pd.DataFrame(rows).T[COLS]
REF = "B7+Ret | size p=2"          # yesterday's W_best_weighted.csv: LB 0.3268
grid["delta vs yesterday's weighted"] = grid["mAP"] - grid.loc[REF, "mAP"]
grid.to_csv(ENS / "weights_results_final.csv")
grid.sort_values("mAP", ascending=False).round(4)

B7 | plain                                         mAP 0.2977  AP_small 0.1455  (47 s)
B7 | plain, RT-DETR x2                             mAP 0.2987  AP_small 0.1410  (45 s)
B7 | size p=2, RT-DETR x2                          mAP 0.2991  AP_small 0.1486  (45 s)
B7+Ret | plain                                     mAP 0.2998  AP_small 0.1435  (50 s)
B7+Ret | plain, RT-DETR x2                         mAP 0.3000  AP_small 0.1401  (49 s)
B7+Ret | size p=2                                  mAP 0.3027  AP_small 0.1476  (49 s)
B7+Ret | size p=2, RT-DETR x1.5                    mAP 0.3027  AP_small 0.1470  (49 s)
B7+Ret | size p=2, RT-DETR x2                      mAP 0.3020  AP_small 0.1468  (49 s)
B7+Ret | size p=2, RT-DETR x3                      mAP 0.2992  AP_small 0.1465  (49 s)
B7+Ret+RT1280 | plain                              mAP 0.3081  AP_small 0.1465  (54 s)
B7+Ret+RT1280 | plain, RT-DETR x2                  mAP 0.3078  AP_small 0.1445  (53 s)
B7+Ret+RT1280 | size p=2                   

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR,delta vs yesterday's weighted
B7+Ret+RT1280+Y26m | size p=2 + FRCNN vote 0.5 +add,0.3138,0.5722,0.3052,0.1525,0.2682,0.5127,0.5250,0.0112
B7+Ret+RT1280+Y26m | size p=2 + FRCNN vote 0.5,0.3138,0.5722,0.3052,0.1525,0.2682,0.5127,0.5249,0.0112
B7+Ret+RT1280+Y26m | size p=2,0.3128,0.5700,0.3044,0.1530,0.2676,0.5102,0.5249,0.0101
B7+Ret+RT1280+Y26m | size p=2 + FRCNN vote 1.0 +add,0.3124,0.5708,0.3033,0.1526,0.2680,0.5133,0.5262,0.0097
B7+Ret+RT1280+Y26m | size p=2 + FRCNN vote 1.0,0.3123,0.5708,0.3032,0.1526,0.2680,0.5133,0.5249,0.0096
"B7+Ret+RT1280+Y26m | size p=2, RT-DETR x1.5",0.3122,0.5695,0.3024,0.1528,0.2649,0.5083,0.5173,0.0095
B7+Ret+RT1280 | size p=2,0.3112,0.5677,0.3035,0.1502,0.2650,0.5095,0.5261,0.0085
B7+Ret+RT1280 -weak | size p=2,0.3107,0.5672,0.3017,0.1545,0.2651,0.5117,0.5208,0.0080
"B7+Ret+RT1280+Y26m | size p=2, RT-DETR x2",0.3107,0.5682,0.3000,0.1523,0.2636,0.5048,0.5126,0.0080
"B7+Ret+RT1280 -weak | size p=2, RT-DETR x1.5",0.3104,0.5674,0.2986,0.1519,0.2616,0.5094,0.5131,0.0077


**Observations:** _fill in_

## 4. Is the best scheme better on most clips, or on one?

mAP per val clip, best scheme vs yesterday's weighted fusion (B7+Ret, size p=2).

In [5]:
BEST = grid["mAP"].idxmax()
best_cfg = CONFIGS[BEST]
print("best:", BEST)
clip_of = {f["image_id"]: f["video"] for f in VAL}
per_clip = []
fused_ref = {i: fuse_frame(CONFIGS[REF], preds, i, VAL_BY_ID[i]["shape"]) for i in LABELLED}
fused_best = {i: fuse_frame(best_cfg, preds, i, VAL_BY_ID[i]["shape"]) for i in LABELLED}
for clip in sorted(set(clip_of.values())):
    ids = [i for i in LABELLED if clip_of[i] == clip]
    per_clip.append({"clip": clip, "frames": len(ids),
                     "yesterday": evaluate_detections(GT_FILE, to_dets(fused_ref, ids), image_ids=ids)["mAP"],
                     "best": evaluate_detections(GT_FILE, to_dets(fused_best, ids), image_ids=ids)["mAP"]})
per_clip = pd.DataFrame(per_clip).assign(delta=lambda t: t["best"] - t["yesterday"])
print(f"best scheme better on {(per_clip.delta > 0).sum()} of {len(per_clip)} clips")
per_clip.round(4)

best: B7+Ret+RT1280+Y26m | size p=2 + FRCNN vote 0.5 +add
best scheme better on 6 of 7 clips


,clip,frames,yesterday,best,delta
0,uav0000086_00000_v,58,0.3157,0.3165,0.0007
1,uav0000117_02622_v,44,0.3166,0.3225,0.0058
2,uav0000137_00458_v,30,0.3460,0.3593,0.0133
3,uav0000182_00000_v,46,0.2370,0.2502,0.0132
4,uav0000268_05773_v,123,0.2000,0.2056,0.0055
5,uav0000305_00000_v,23,0.4270,0.4198,-0.0072
6,uav0000339_00001_v,35,0.3146,0.3177,0.0030


**Observations:** _fill in_

## 5. Best scheme on every val and test frame

Saved as flat detection lists with clip and frame number (`fused_best_valall.pkl`, `fused_best_test.pkl`): `temporal_v2.ipynb` uses them as its ensemble input. Also writes `submissions/W_final.csv`. Needs the test predictions from `ensemble_temporal.ipynb` section 5.

In [6]:
def fuse_all(split, frames, preds_split):
    t0 = time.time()
    dets = []
    for f in frames:
        xyxy, s, c = fuse_frame(best_cfg, preds_split, f["image_id"], f["shape"])
        dets += [{"image_id": f["image_id"], "video": f["video"], "frame": f["frame"], "category_id": int(k) + 1,
                  "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(v)}
                 for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
    (ENS / f"fused_best_{split}.pkl").write_bytes(pickle.dumps(dets))
    print(f"{split}: fused {len(frames)} frames in {(time.time() - t0) / 60:.1f} min")
    return dets


needed = set(best_cfg["members"]) | ({FRCNN} if best_cfg.get("vote") else set())
(ENS / "weights_best.json").write_text(json.dumps({"label": BEST, **best_cfg}, indent=1))
_ = fuse_all("valall", VAL, preds)   # every val frame, for temporal_v2.ipynb
del preds, _
test_preds = {n: pickle.loads((ENS / f"test_{n}.pkl").read_bytes()) for n in needed}
test_dets = fuse_all("test", TEST, test_preds)
rows_by_id = {}
for d in test_dets:
    x, y, w, h = d["bbox"]
    rows_by_id.setdefault(d["image_id"], []).append((d["category_id"] - 1, d["score"], x, y, w, h))
sub = write_submission(rows_by_id, TEST_CSV, ENS / "submissions" / "W_final.csv")
print(f"wrote W_final.csv: {len(sub)} rows")

valall: fused 2846 frames in 3.7 min
test: fused 3811 frames in 4.9 min
wrote W_final.csv: 3811 rows
